In [ ]:
"""
Environment Setup Module.
Installs the necessary libraries for this lab. Run this cell first.
"""
!pip install qiskit[visualization] qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc

import math
import fractions
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from IPython.display import display

print("\n ENVIRONMENT SETUP COMPLETE — proceed to the next cell.")

---
## Exercise 1: Build and Verify the Quantum Fourier Transform

The QFT is the quantum analogue of the DFT. On an $n$-qubit basis state $|x\rangle$:

$$\text{QFT}_N |x\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i xk/N} |k\rangle, \qquad N = 2^n.$$

**Circuit structure** (for $n$ qubits):
1. For each qubit $i$ from 0 to $n-1$:
   - Apply $H$ to qubit $i$.
   - Apply controlled-$R_k$ gates: $R_k = P(2\pi/2^k)$ with qubit $j > i$ as control.
2. Reverse qubit order with SWAP gates (bit-reversal permutation).

**Your Task:**
1. Run the QFT on all computational basis states $|0\rangle, |1\rangle, |2\rangle, |3\rangle$ of a 2-qubit register.
2. Verify that the output state for $|0\rangle$ is the uniform superposition.
3. Observe how the phase pattern changes for $|1\rangle$, $|2\rangle$, $|3\rangle$.

In [ ]:
"""
QFT Circuit Module.
Implements the Quantum Fourier Transform for an arbitrary number of qubits
and verifies it on all 2-qubit basis states via the statevector simulator.
"""

def build_qft(n: int) -> QuantumCircuit:
    """
    Constructs the n-qubit QFT circuit.

    Decomposition:
        For each qubit i:
            H(i)
            CP(pi/2^(j-i), control=j, target=i)  for j = i+1 ... n-1
        Followed by SWAP pairs for bit-reversal.

    Args:
        n: Number of qubits.

    Returns:
        QuantumCircuit implementing QFT_N (N = 2^n).
    """
    qc = QuantumCircuit(n, name='QFT')
    for i in range(n):
        qc.h(i)
        for j in range(i + 1, n):
            angle = math.pi / (2 ** (j - i))
            qc.cp(angle, j, i)      # controlled phase R_{j-i+1}
    # Bit-reversal permutation
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    return qc


# ── Visualise the 2-qubit QFT circuit ──────────────────────────────────────
qft2 = build_qft(2)
print("2-Qubit QFT Circuit:")
display(qft2.draw('mpl', style='iqp'))

# ── Verify on all 2-qubit basis states ─────────────────────────────────────
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

print("\nQFT output statevectors for 2-qubit input states:")
print("-" * 62)

for x in range(4):
    # Build |x> state
    qc = QuantumCircuit(2)
    if x & 1: qc.x(0)   # bit 0 (LSB in Qiskit = q0)
    if x & 2: qc.x(1)   # bit 1
    qc.compose(qft2, inplace=True)

    sv = Statevector.from_instruction(qc)
    amps = sv.data
    amp_str = '  '.join(f'{a: .3f}' for a in amps)
    print(f"  QFT|{x:02b}> = [ {amp_str} ]")

print()
print("Expected for |00>: all amplitudes = 0.5 + 0i  (uniform superposition)")
print("Expected for |01>: amplitudes 0.5, 0.5i, -0.5, -0.5i  (linear phase ramp)")

---
## Exercise 2: Full Shor's Algorithm — $N = 15$, $a = 7$

### Circuit Design

**Registers:**
- **Counting register** `count[0..3]`: 4 qubits, 16 states.
- **Work register** `work[0..1]`: 2 qubits, encoding the orbit $\{1, 7, 4, 13\}$
  as $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$ respectively.

**Modular exponentiation** reduces to controlled cyclic shifts of the work register:

| Counting qubit | $j$ | $U^{2^j}$ | Effect on work | Gate |
|---|---|---|---|---|
| `count[0]` | 0 | $U^1$ | add 1 mod 4 | CCNOT(c0,w0,w1) + CNOT(c0,w0) |
| `count[1]` | 1 | $U^2$ | add 2 mod 4 | CNOT(c1,w1) |
| `count[2]` | 2 | $U^4 = I$ | identity | (none) |
| `count[3]` | 3 | $U^8 = I$ | identity | (none) |

**Student Experiment (Over-rotation):**  
The IQFT must be applied exactly once. What happens if you apply it twice?  
Change `EXTRA_IQFT` to `True` and observe the distribution flatten.

**Expected output:** Peaks at $\{|0000\rangle, |0100\rangle, |1000\rangle, |1100\rangle\}$  
(decimal values $0, 4, 8, 12$), each with probability $\approx 25\%$.

In [ ]:
"""
Shor Order-Finding Module.
Implements the period-finding circuit for a=7, N=15 using a 2-qubit
orbit-encoded work register and the QFT built in Exercise 1.
"""

# ── STUDENT EXPERIMENT ZONE ────────────────────────────────────────────────
EXTRA_IQFT = False   # Set to True to observe the over-rotation failure mode
N_SHOTS    = 4000
# ──────────────────────────────────────────────────────────────────────────


def build_shor_N15_a7(extra_iqft: bool = False) -> QuantumCircuit:
    """
    Constructs the order-finding circuit for a=7, N=15.

    Work-register encoding:
        |00> <-> 1 (the identity element of the orbit)
        |01> <-> 7
        |10> <-> 4
        |11> <-> 13

    Controlled-U^1 (add 1 mod 4):
        New w1 = w1 XOR (ctrl AND w0)  =>  CCNOT(ctrl, w0, w1)
        New w0 = w0 XOR ctrl           =>  CNOT(ctrl, w0)

    Controlled-U^2 (add 2 mod 4 = flip w1):
        New w1 = w1 XOR ctrl           =>  CNOT(ctrl, w1)

    Args:
        extra_iqft: If True, apply the IQFT a second time (demonstrates failure).

    Returns:
        A QuantumCircuit ready for simulation.
    """
    C  = QuantumRegister(4, 'count')
    W  = QuantumRegister(2, 'work')
    cr = ClassicalRegister(4, 'c')
    qc = QuantumCircuit(C, W, cr)

    # ── 1. Superposition on counting register ─────────────────────────────
    qc.h(C)
    qc.barrier(label='H')

    # ── 2. Controlled modular exponentiation ──────────────────────────────
    # c-U^1: controlled by C[0]
    qc.ccx(C[0], W[0], W[1])
    qc.cx(C[0], W[0])
    # c-U^2: controlled by C[1]
    qc.cx(C[1], W[1])
    # c-U^4 = I (C[2]) and c-U^8 = I (C[3]): no gates needed
    qc.barrier(label='Oracle')

    # ── 3. Inverse QFT on counting register ───────────────────────────────
    iqft = build_qft(4).inverse()
    iqft.name = 'IQFT'
    qc.compose(iqft, qubits=C, inplace=True)
    qc.barrier(label='IQFT')

    if extra_iqft:
        qc.compose(iqft, qubits=C, inplace=True)   # deliberate over-rotation
        qc.barrier(label='IQFT×2')

    # ── 4. Measurement ────────────────────────────────────────────────────
    qc.measure(C, cr)
    return qc


def extract_period(y: int, n_count: int, N: int = 15) -> int:
    """
    Continued-fractions algorithm: find the period r from measurement y.

    Args:
        y:       Integer value of the measured bitstring.
        n_count: Number of counting qubits (2^n_count is the QFT size).
        N:       The number we are factoring.

    Returns:
        Period estimate r_hat (the denominator of the best rational approximation
        to y / 2^n_count with denominator <= N).
    """
    phase = y / (2 ** n_count)
    if phase == 0:
        return 1   # trivial; caller should discard
    return fractions.Fraction(phase).limit_denominator(N).denominator


# ── Build and draw circuit ──────────────────────────────────────────────────
qc_shor = build_shor_N15_a7(extra_iqft=EXTRA_IQFT)
display(qc_shor.draw('mpl', style='iqp', fold=40))

# ── Simulation ─────────────────────────────────────────────────────────────
simulator = AerSimulator()
job       = simulator.run(qc_shor, shots=N_SHOTS)
counts    = job.result().get_counts()

print(f"\nShor Order-Finding Results (a=7, N=15, {N_SHOTS} shots)")
print(f"{'State':>8}   {'y':>3}   {'phi':>7}   {'r_hat':>6}   {'7^r mod 15':>11}   {'Prob':>6}")
print("-" * 62)

for state, n in sorted(counts.items(), key=lambda x: -x[1]):
    y      = int(state, 2)
    phi    = fractions.Fraction(y, 2**4).limit_denominator(15)
    r_hat  = phi.denominator
    check  = pow(7, r_hat, 15)
    prob   = n / N_SHOTS * 100
    mark   = '✓' if check == 1 and r_hat > 1 else '✗'
    print(f"  |{state}>   {y:3d}   {str(phi):>7}   {r_hat:>6}   {check:>11}  {mark}  {prob:5.1f}%")

display(plot_histogram(counts, title=f"Shor N=15, a=7{' (extra IQFT!)' if EXTRA_IQFT else ''}",
                        figsize=(10, 4)))

---
## Exercise 3: Classical Post-Processing — Factor Extraction

Once the period $r$ is found, the classical steps are:
1. Check $r$ is **even**.
2. Check $a^{r/2} \not\equiv -1 \pmod{N}$.
3. Compute $p = \gcd(a^{r/2} - 1, N)$ and $q = \gcd(a^{r/2} + 1, N)$.

Run the cell below for a complete trace of all four possible measurement outcomes.

In [ ]:
"""
Factor Extraction Module.
Demonstrates the full classical post-processing pipeline for each
possible measurement outcome of the Shor circuit.
"""

a, N = 7, 15

print(f"Factor extraction trace for a = {a}, N = {N}")
print("=" * 68)

for y in [0, 4, 8, 12]:
    r_hat  = extract_period(y, 4, N)
    # Try r_hat and small multiples
    r_final = None
    for m in range(1, 8):
        r_try = r_hat * m
        if pow(a, r_try, N) == 1:
            r_final = r_try
            break

    print(f"\n  y = {y}  →  phi = {y}/{2**4} = {fractions.Fraction(y,16)}  →  r_hat = {r_hat}")

    if y == 0 or r_hat == 1:
        print("    ⚠  Trivial outcome — discard and re-run.")
        continue

    if r_final is None:
        print("    ✗  No valid period found (unusual — retry).")
        continue

    print(f"    Verified: 7^{r_final} mod 15 = {pow(a,r_final,N)}  →  r = {r_final}")

    if r_final % 2 != 0:
        print("    ✗  r is odd — conditions not met, retry.")
        continue

    half = pow(a, r_final // 2, N)
    if half == N - 1:
        print(f"    ✗  a^(r/2) = {half} ≡ -1 (mod {N}) — conditions not met, retry.")
        continue

    p = math.gcd(half - 1, N)
    q = math.gcd(half + 1, N)
    print(f"    gcd(7^{r_final//2} - 1, 15) = gcd({half-1}, 15) = {p}")
    print(f"    gcd(7^{r_final//2} + 1, 15) = gcd({half+1}, 15) = {q}")
    if p * q == N and p > 1 and q > 1:
        print(f"    ✓  FACTORS FOUND: {N} = {p} × {q}")
    else:
        print(f"    ✗  Trivial factors ({p}, {q}) — retry.")

print("\n" + "=" * 68)
print("Success probability per circuit run: ~75% (y ∈ {4, 12} → direct; y=8 → double r_hat)")

---
## OPTIONAL: Execute on a Real Quantum Computer (QPU)

**Context:** Shor's Algorithm is extremely demanding on real hardware. The circuit
requires Toffoli (CCX) gates, which decompose into many single and two-qubit gates,
each introducing noise. On current NISQ devices, we expect the theoretical $\approx 25\%$
peaks to be substantially degraded.

**Step 1:** Get your API token from
[quantum.ibm.com](https://quantum.ibm.com) and paste it below.

In [ ]:
"""
IBM Quantum Authentication Module.
"""
from qiskit_ibm_runtime import QiskitRuntimeService

# ── STUDENT INPUT REQUIRED ────────────────────────────────────────────────
TOKEN = "YOUR_IBM_QUANTUM_TOKEN_HERE"
# ─────────────────────────────────────────────────────────────────────────

def check_ibm_quota(api_token: str) -> None:
    if api_token == "YOUR_IBM_QUANTUM_TOKEN_HERE":
        print(" ACTION REQUIRED: Insert your personal API token above.")
        return
    print("Authenticating...")
    try:
        QiskitRuntimeService.save_account(
            channel="ibm_quantum", token=api_token,
            set_as_default=True, overwrite=True
        )
        service = QiskitRuntimeService()
        print(" Authentication successful!")
        print(f"Available backends: {[b.name for b in service.backends(operational=True, simulator=False)]}")
    except Exception as e:
        print(f" Authentication failed: {e}")

check_ibm_quota(TOKEN)

**Step 2:** Transpile and run on the least-busy backend. The CCX gate will
be decomposed into native gates (typically CX + single-qubit rotations).
Hardware noise will partially flatten the theoretical 25% peaks.

In [ ]:
"""
QPU Execution Module.
Transpiles the Shor circuit for a real backend and analyses the noise impact.
"""
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2

def run_shor_on_hardware() -> None:
    try:
        service = QiskitRuntimeService()
    except Exception:
        print(" Please run the authentication cell first.")
        return

    print("Finding least-busy quantum computer...")
    backend = service.least_busy(operational=True, simulator=False)
    print(f" Selected: {backend.name} ({backend.num_qubits} qubits)")

    qc_hw = build_shor_N15_a7(extra_iqft=False)

    print("Transpiling...")
    pm         = generate_preset_pass_manager(backend=backend, optimization_level=3)
    isa_circuit = pm.run(qc_hw)
    print(f" Transpiled gate count: {isa_circuit.count_ops()}")

    print(f"Submitting to {backend.name} queue...")
    sampler = SamplerV2(backend)
    job     = sampler.run([isa_circuit], shots=2000)
    print(f" Job ID: {job.job_id()} — waiting for results...")

    result = job.result()
    counts = result[0].data.c.get_counts()

    total = sum(counts.values())
    target_hits = sum(counts.get(s, 0) for s in ['0100','1000','1100'])
    accuracy = target_hits / total * 100

    print(f"\n Real Hardware Results ({backend.name}):")
    for state, n in sorted(counts.items(), key=lambda x: -x[1])[:8]:
        print(f"  |{state}>  {n:4d} shots ({n/total*100:.1f}%)")

    print(f"\n=== NOISE ANALYSIS ===")
    print(f"Ideal probability at peaks {{4,8,12}}: ~75.0%")
    print(f"Actual hardware accuracy:           {accuracy:.1f}%")
    print(f"Noise degradation: {75.0 - accuracy:.1f} percentage points")
    print("The CCX gate decomposes into ~6 CX gates, each introducing ~0.5-1% error.")
    print("=======================")

    display(plot_histogram(counts,
                           title=f"Shor N=15, a=7 on {backend.name} (real QPU)",
                           figsize=(10, 4)))

if TOKEN != "YOUR_IBM_QUANTUM_TOKEN_HERE":
    run_shor_on_hardware()
else:
    print(" Skipped: insert your IBM Quantum API token in the cell above.")